In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from joblib import Parallel, delayed
from datetime import datetime
import sys
from pathlib import Path

import sys
# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))

# from indicators.indicators import load_symbol, build_indicators
from mdt.mdt import load_symbol
from backtester import engine

import vars

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 40)
pd.set_option('display.max_colwidth', 200)

color_back = "#0d1117"

path_repo = vars.path_repo
path_data = vars.path_data

In [2]:
symbs = vars.symbs

symbs_crypto = [x for x in symbs if "-USD" in x]
symbs_forex = [x for x in symbs if "=X" in x]
symbs_stocks = [x for x in symbs if x not in symbs_crypto + symbs_forex]

print("Total:", len(symbs))
print("Crypto :", len(symbs_crypto), f"({(len(symbs_crypto)/len(symbs))*100:.0f}%)")
print("Forex:", len(symbs_forex), f"({(len(symbs_forex)/len(symbs))*100:.0f}%)")
print("Stocks:", len(symbs_stocks), f"({(len(symbs_stocks)/len(symbs))*100:.0f}%)")

Total: 12
Crypto : 3 (25%)
Forex: 3 (25%)
Stocks: 6 (50%)


In [3]:
symb = "ETH-USD"
df = load_symbol(symb, "1h", 400) 
df

Price,Close,High,Low,Open,Volume,date,EMA20,EMA50,EMA200,RSI,MACD,MACD_signal,MACD_hist,BB_upper,BB_mid,BB_lower,BB_position,STOCH_K,STOCH_D,ATR,OBV,OBV_EMA,ADX,ADX+DI,ADX-DI,prev26,prev52,VPVMA,VPVMAS,VWAP,spread,vol_ma,volume_ma,volume_increasing,order_flow,volatility,CMF,PSAR,WillR,Donchian_high,Donchian_low,ema50_distance,MACD_bearish_div,MACD_bullish_div,sign_macd_rsi_robust,sign_macd_rsi_simple,sign_macd_rsi_simple_break,sign_macd_rsi_simple_bb_pos,sign_macd_rsi_bb,sign_macd_rsi_bb_ema,ind_adx,sign_adx_crec,sign_adx_crec_rsi,sign_adx_rsi,sign_adx_rsi_robust,sign_adx_rsi_xtrem,sign_vpvma,sign_sinergia,sign_rsi_bb_rever,sign_ma_macd,str_squ_kel_bb,sign_gold,ind_3ema,sign_rsi30,sign_rsi20,sign_rsi15,trend_adx,sign_pullb_ema,sign_rsi_div,ind_vwap_deviation,sign_supo_resi,ind_volume_anomaly,ind_compress_range,sign_sobreextema,sign_stochastic,sign_cmf,sign_elderray,sign_willr,sign_donchian,sign_liquidsweep,sign_macd_div1,sign_macd_div3,sign_macd_div5,sign_random,sign_random2,sign_random3
Datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025-09-15 08:00:00+00:00,4514.619141,4599.745117,4513.007324,4599.745117,3385788416,2025-09-15 08:00:00+00:00,4618.462508,4623.463791,4503.260075,29.270522,-12.931310,-7.684653,-5.246657,4673.247026,4613.626978,4554.006929,-0.330323,25.104345,53.625985,31.859328,1.884071e+10,2.291256e+10,28.452089,11.508392,41.204295,4667.408203,4748.386230,18871.911685,2910.926278,4577.698976,0.019213,651871232.0,651871232.0,True,sell,0.005610,-0.402502,4670.205078,-98.974657,4670.205078,4513.007324,-0.023542,False,False,0,0,0,0,1,1,-1,-1,0,0,0,0,0,0,1,0,-1,-1,0,1,0,0,-2.0,0,0,0,-1,1,0,0,0,-1,0,0,-1,0,0,0,0,0,0,0
2025-09-15 09:00:00+00:00,4525.980469,4535.060059,4507.732422,4512.894531,3105804288,2025-09-15 09:00:00+00:00,4609.654695,4619.640916,4503.486148,32.204481,-18.672161,-9.882155,-8.790007,4680.617536,4609.262378,4537.907220,-0.083573,9.974137,30.999113,31.535636,2.194651e+10,2.282056e+10,30.517333,10.796052,39.848627,4652.829102,4725.704590,14588.247069,4532.201554,4563.263398,0.006038,807161446.4,807161446.4,True,buy,0.005615,-0.260517,4663.917168,-88.768543,4670.205078,4507.732422,-0.020274,False,False,0,0,0,0,0,0,-1,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-2.0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0
2025-09-15 10:00:00+00:00,4532.098145,4536.320312,4510.450195,4527.214355,957607936,2025-09-15 10:00:00+00:00,4602.268357,4616.207866,4503.770845,33.796959,-22.469172,-12.399558,-10.069614,4684.884562,4605.604150,4526.323739,0.036418,9.084538,14.721007,31.130956,2.290412e+10,2.282851e+10,32.367244,10.444382,37.483300,4657.789551,4712.777832,10664.372529,5518.784305,4560.793843,0.005708,716928204.8,716928204.8,True,buy,0.005634,-0.188231,4654.546083,-85.003186,4670.205078,4507.732422,-0.018221,False,False,0,0,0,0,0,0,-1,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2025-09-15 11:00:00+00:00,4534.335449,4540.383789,4525.642090,4531.789551,429432832,2025-09-15 11:00:00+00:00,4595.798556,4612.997183,4504.074970,34.403800,-25.009504,-14.921547,-10.087957,4688.153475,4603.214307,4518.275138,0.094540,14.200707,11.086461,29.960295,2.333355e+10,2.287661e+10,33.855759,11.046082,36.165919,4667.737793,4714.285156,5560.402853,6221.312976,4559.885909,0.003251,738399846.4,738399846.4,False,buy,0.005550,-0.177540,4645.737264,-83.626151,4670.205078,4507.732422,-0.017052,False,False,0,0,0,0,0,0,-1,-1,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,-2.0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
2025-09-15 12:00:00+00:00,4515.225098,4537.743164,4514.793457,4533.252441,0,2025-09-15 12:00:00+00:00,4588.124893,4609.162983,4504.185917,31.728489,-28.239257,-17.585089,-10.654168,4692.722991,4599.093164,4505.463337,0.052130,11.994106,11.759784,29.459539,2.333355e+10,2.292013e+10,35.424160,10.431427,36.783876,4662.393555,4731.202637,3623.928692,6721.696257,4559.885909,0.005083,738399846.4,738399846.4,False,sell,0.005529,-0.177540,4637.456973,-95.388347,4670.205078,4507.732422,-0.020381,False

In [ ]:
# cols_sign = [col for col in df.columns if col.startswith("sign_")] 
# cols_indic = [col for col in df.columns if not col.startswith("sign_")] 

cols_sign = [
'sign_random', 'sign_random2', 
'sign_macd_rsi_robust', 'sign_macd_rsi_simple', #'sign_macd_rsi_simple_break', #'sign_macd_rsi_simple_bb_pos', 'sign_macd_rsi_bb', 'sign_macd_rsi_bb_ema', 
'sign_adx_crec', 'sign_adx_crec_rsi', #'sign_adx_rsi', 'sign_adx_rsi_robust', #'sign_adx_rsi_xtrem', 
'sign_vpvma', 'sign_sinergia', #'sign_rsi_bb_rever', #'sign_ma_macd', 'str_squ_kel_bb', 
'sign_gold', 
'sign_rsi30', #'sign_rsi20', 'sign_rsi15', 'sign_pullb_ema', 'sign_rsi_div', 'sign_supo_resi', 'sign_sobreextema', 'sign_stochastic', 'sign_cmf', 'sign_elderray', 
# 'sign_willr', 'sign_donchian', 'sign_liquidsweep', 'sign_macd_div1', 'sign_macd_div3', 'sign_macd_div5', 'sign_random3'
]
cols_indic = [
'Close', 'High', 'Low', 'Open', 'Volume', 'date', 'EMA20', 'EMA50', 'EMA200', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_upper', 'BB_mid', 'BB_lower', 'BB_position', 
'STOCH_K', 'STOCH_D', 'ATR', 'OBV', 'OBV_EMA', 'ADX', 'ADX+DI', 'ADX-DI', 'prev26', 'prev52', 'VPVMA', 'VPVMAS', 'VWAP', 'spread', 'vol_ma', 'volume_ma', 'volume_increasing', 
'order_flow', 'volatility', 'CMF', 'PSAR', 'WillR', 'Donchian_high', 'Donchian_low', 'ema50_distance', 'MACD_bearish_div', 'MACD_bullish_div', 'ind_volume_anomaly', 'ind_compress_range', 
'ind_vwap_deviation', 'ind_3ema', 'ind_adx', 'trend_adx',
]

order = sorted(cols_sign)
# df = df[order]

print("Signals   :", len(cols_sign))
print("Indicators:", len(cols_indic))

Signals   : 10
Indicators: 50


In [5]:
resume, trades, equity = engine(df, "sign_macd_div1", symb)

In [6]:
def symbol_performance(df, signal, symb):
    resume, trades, equity = engine(df, signal, symb)
    resume["signal"] = signal
    resume["symbol"] = symb
    trades["signal"] = signal
    trades["symbol"] = symb
    equity["signal"] = signal
    equity["symbol"] = symb

    return resume, trades, equity

results = Parallel(n_jobs=-1, backend="threading")(delayed(symbol_performance)(df, signal, symb) for signal in cols_sign)

resume =  []
trades = []
equity = []

for _ax1, _ax2, _ax3 in results:
    resume.append(_ax1)
    trades.append(_ax2)
    equity.append(_ax3)
    
resume = pd.concat(resume).sort_values("capital_final", ascending = False)
print(len(resume))

10


In [10]:
tmp = list(set(df["date"].astype(str).str[:10].tolist()))
total = len(tmp)
total

390

In [11]:
resume["trades_byday"] = resume["trades"] / total

In [12]:
resume

,symbol,signal,capital,capital_final,pnl,return,trades,win_rate,avg_win,avg_loss,expectancy,profit_factor,avg_trade,avg_return,median_return,max_drawdown,sharpe,sortino,cagr,calmar,recovery_factor,var_95,cvar_95,exposure,avg_bars,max_loss_streak,max_win_streak,n_tp,n_sl,n_ts,n_rev,fees,open_trade,trades_byday
0,ETH-USD,sign_adx_crec_rsi,1000,966.746219,-33.253781,-0.033254,3,0.000000,0.000000,-10.645751,-10.645751,0.000000,-10.645751,-1.717423,-1.762869,-0.036982,-1.604513,-0.112721,-0.031261,-0.845302,-0.899191,0.000000,-0.000006,0.003335,11.333333,3,0,0,3,0,0,2.640298,False,0.007692
0,ETH-USD,sign_macd_rsi_simple,1000,927.393569,-72.606431,-0.072606,29,0.344828,14.002219,-10.558195,-2.089087,0.697997,-2.089087,-0.375540,-1.586278,-0.094460,-1.090293,-0.289334,-0.068340,-0.723478,-0.768644,0.000000,-0.000054,0.036475,12.689655,8,4,10,19,0,0,24.067119,False,0.074359
0,ETH-USD,sign_macd_rsi_robust,1000,893.876832,-106.123168,-0.106123,41,0.341463,13.664918,-10.369078,-2.162348,0.683331,-2.162348,-0.488426,-1.456366,-0.127248,-1.350290,-0.493484,-0.099996,-0.785837,-0.833989,0.000000,-0.000080,0.059608,14.512195,6,4,14,27,0,0,34.970706,False,0.105128
0,ETH-USD,sign_gold,1000,690.841927,-309.158073,-0.309158,183,0.382514,11.975425,-9.395742,-1.220979,0.789550,-1.220979,-0.247848,-1.020053,-0.328711,-1.919376,-1.430015,-0.293423,-0.892649,-0.940516,-0.002533,-0.004907,0.302130,16.344262,9,5,69,78,0,36,171.361941,False,0.469231
0,ETH-USD,sign_sinergia,1000,609.827172,-390.172828,-0.390173,297,0.390572,11.289572,-8.626286,-0.847702,0.838751,-0.847702,-0.147218,-0.984372,-0.439156,-2.194488,-2.116904,-0.371528,-0.846004,-0.888460,-0.003327,-0.005660,0.386916,13.107744,10,4,115,156,0,26,276.777385,False,0.761538
0,ETH-USD,sign_adx_crec,1000,574.813145,-425.186855,-0.425187,322,0.385093,12.071561,-9.072429,-0.930023,0.833290,-0.930023,-0.194997,-1.364663,-0.499730,-2.341214,-2.401290,-0.405476,-0.811389,-0.850832,-0.003722,-0.005876,0.461158,14.282609,8,8,124,190,0,8,240.379348,True,0.825641
0,ETH-USD,sign_rsi30,1000,511.020758,-488.979242,-0.488979,286,0.353147,9.496996,-7.389769,-1.426261,0.701625,-1.426261,-0.340249,-1.413661,-0.501292,-2.958753,-2.512111,-0.467656,-0.932902,-0.975439,-0.003460,-0.006332,0.426189,14.818182,11,5,92,185,0,9,170.320681,True,0.733333
0,ETH-USD,sign_random2,1000,509.500852,-490.499148,-0.490499,384,0.398438,8.679624,-7.204332,-0.875568,0.797970,-0.875568,-0.177888,-0.635048,-0.553808,-2.979928,-2.864881,-0.469143,-0.847122,-0.885684,-0.003441,-0.005884,0.430923,11.416667,11,8,97,174,0,113,305.018279,True,0.984615
0,ETH-USD,sign_random,1000,377.730716,-622.269284,-0.622269,653,0.425727,6.939616,-6.159830,-0.583037,0.835180,-0.583037,-0.144004,-0.286107,-0.672591,-3.581605,-4.029387,-0.599199,-0.890882,-0.925182,-0.004358,-0.006857,0.561868,8.989280,9,7,123,224,0,306,484.821831,True,1.674359
0,ETH-USD,sign_vpvma,1000,122.070425,-877.929575,-0.877930,1320,0.423485,2.986637,-2.973565,-0.449510,0.737789,-0.449510,-0.142615,-0.188150,-0.880835,-6.512313,-7.903164,-0.861251,-0.977767,-0.996701,-0.005297,-0.008240,0.678933,5.777273,12,9,100,225,0,995,568.784556,True,3.384615


In [ ]:
trades

[     trade_id                entry_time                 exit_time  side  \
 0           0 2025-09-13 10:00:00+00:00 2025-09-13 12:00:00+00:00     1   
 1           1 2025-09-13 13:00:00+00:00 2025-09-13 16:00:00+00:00    -1   
 2           2 2025-09-14 08:00:00+00:00 2025-09-14 10:00:00+00:00     1   
 3           3 2025-09-14 11:00:00+00:00 2025-09-14 14:00:00+00:00    -1   
 4           4 2025-09-14 17:00:00+00:00 2025-09-14 22:00:00+00:00     1   
 ..        ...                       ...                       ...   ...   
 639       639 2026-10-05 15:00:00+00:00 2026-10-05 16:00:00+00:00     1   
 640       640 2026-10-05 17:00:00+00:00 2026-10-05 19:00:00+00:00    -1   
 641       641 2026-10-05 20:00:00+00:00 2026-10-06 03:00:00+00:00     1   
 642       642 2026-10-06 04:00:00+00:00 2026-10-06 06:00:00+00:00    -1   
 643       643 2026-10-06 07:00:00+00:00 2026-10-07 01:00:00+00:00     1   
 
      entry_price   exit_price      size     margin   sl_initial     sl_final  \
 0   

In [ ]:
equity

[                          time       equity         cash  position  \
 0    2025-09-13 08:00:00+00:00  1000.000000  1000.000000         0   
 1    2025-09-13 09:00:00+00:00  1000.000000  1000.000000         0   
 2    2025-09-13 10:00:00+00:00   996.370057   990.845790         1   
 3    2025-09-13 11:00:00+00:00   996.953682   990.845790         1   
 4    2025-09-13 12:00:00+00:00   996.050087   996.050087         0   
 ...                        ...          ...          ...       ...   
 9292 2026-10-06 23:00:00+00:00   336.996453   333.029495         1   
 9293 2026-10-07 00:00:00+00:00   336.909422   333.029495         1   
 9294 2026-10-07 01:00:00+00:00   333.285777   333.285777         0   
 9295 2026-10-07 02:00:00+00:00   333.285777   333.285777         0   
 9296 2026-10-07 03:00:00+00:00   333.285777   333.285777         0   
 
             close         peak  drawdown       signal   symbol  
 0     4731.202637  1000.000000  0.000000  sign_random  ETH-USD  
 1     4732.58

In [ ]:

# Agregar para seleccionar señales

score =
0.30 * rank_sharpe
+ 0.20 * rank_return
+ 0.20 * rank_profit_factor
+ 0.15 * rank_calmar
+ 0.15 * rank_expectancy

Tambien penalizar por muchos trades

if trades < 50:
    score *= 0.5

if max_drawdown < -0.30:
    score *= 0.5

if profit_factor < 1.1:
    score *= 0.5


SyntaxError: invalid syntax (4024874182.py, line 3)

In [ ]:
qewe

In [ ]:
# Walk Forward + OOS

INITIAL_CAPITAL = 1000

RISK_PER_TRADE = 0.01
FEE_RATE = 0.0007
SLIPPAGE_PCT = 0.0005

ATR_SL = 2
MULTIPLIER_TP = 1.5

LEVERAGE = 100

USE_TRAILING_STOP = False
USE_BREAKEVEN = False

TRAILING_ATR_MULT = 1
TRAILING_ACTIVATION_R = 2.5
BREAKEVEN_TRIGGER_R = 3

CLOSE_ON_REVERSE = True

MIN_TRADES = 30


# ============================================================
# LISTA DE ESTRATEGIAS
# ============================================================

STRATEGY_COLUMNS = [
    "sign_macd_rsi_robust",
    "str_macd_rsi_simple",
    
]


# ============================================================
# MÉTRICAS DE UNA ESTRATEGIA
# ============================================================

def calculate_strategy_score(resume):

    """
    Convierte las métricas del backtest en un score.

    IMPORTANTE:
    No utilizamos solamente Return.

    Una estrategia que gana 100% pero tiene un drawdown
    enorme no necesariamente es mejor que una que gana 40%
    de forma estable.
    """

    if resume is None:
        return -np.inf

    trades = resume.get("trades", 0)

    if trades < MIN_TRADES:
        return -np.inf

    sharpe = resume.get("sharpe", 0)
    sortino = resume.get("sortino", 0)
    profit_factor = resume.get("profit_factor", 0)
    total_return = resume.get("total_return", 0)
    max_drawdown = resume.get("max_drawdown", 0)
    calmar = resume.get("calmar", 0)

    # --------------------------------------------------------
    # Protección contra NaN / infinitos
    # --------------------------------------------------------

    values = [
        sharpe,
        sortino,
        profit_factor,
        total_return,
        max_drawdown,
        calmar
    ]

    values = [
        0 if pd.isna(x) or np.isinf(x) else x
        for x in values
    ]

    sharpe, sortino, profit_factor, total_return, max_drawdown, calmar = values

    # --------------------------------------------------------
    # Penalización por drawdown
    # --------------------------------------------------------

    drawdown_penalty = abs(max_drawdown)

    # --------------------------------------------------------
    # Score
    # --------------------------------------------------------

    score = (
        0.25 * sharpe
        + 0.15 * sortino
        + 0.20 * profit_factor
        + 0.20 * total_return
        + 0.10 * calmar
        - 0.10 * drawdown_penalty
    )

    return score


# ============================================================
# NORMALIZAR RESULTADO DEL ENGINE
# ============================================================

def extract_metric(resume, key, default=0):

    if resume is None:
        return default

    value = resume.get(key, default)

    if value is None:
        return default

    if pd.isna(value):
        return default

    if np.isinf(value):
        return default

    return value


# ============================================================
# WALK FORWARD
# ============================================================

def walk_forward(
    df,
    strategy_columns,
    train_months=12,
    validation_months=3,
    step_months=3,
    top_n=5,
    min_trades=30
):

    """
    Walk Forward para seleccionar estrategias.

    Ejemplo:

    Train:
        enero 2024 - diciembre 2024

    Validation:
        enero 2025 - marzo 2025

    Luego:

    Train:
        abril 2024 - marzo 2025

    Validation:
        abril 2025 - junio 2025

    etc.

    La estrategia NO se selecciona mirando el futuro.

    """

    df = df.copy()

    # --------------------------------------------------------
    # Validaciones
    # --------------------------------------------------------

    if "time" not in df.columns:

        if isinstance(df.index, pd.DatetimeIndex):

            df = df.reset_index()

            if "time" not in df.columns:
                df = df.rename(columns={
                    df.columns[0]: "time"
                })

        else:
            raise ValueError(
                "El DataFrame debe tener columna 'time'"
            )

    df["time"] = pd.to_datetime(df["time"])

    df = df.sort_values("time").reset_index(drop=True)

    # --------------------------------------------------------
    # Fechas disponibles
    # --------------------------------------------------------

    min_date = df["time"].min()
    max_date = df["time"].max()

    print("=" * 70)
    print("WALK FORWARD")
    print("=" * 70)

    print("Inicio:", min_date)
    print("Fin:", max_date)

    # --------------------------------------------------------
    # Resultados
    # --------------------------------------------------------

    all_results = []

    selected_strategies = []

    # --------------------------------------------------------
    # Primer train
    # --------------------------------------------------------

    train_start = min_date

    while True:

        train_end = (
            train_start
            + pd.DateOffset(months=train_months)
        )

        validation_end = (
            train_end
            + pd.DateOffset(months=validation_months)
        )

        # ----------------------------------------------------
        # Si no existe suficiente futuro
        # ----------------------------------------------------

        if validation_end > max_date:

            break

        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        train_mask = (
            (df["time"] >= train_start)
            &
            (df["time"] < train_end)
        )

        train_df = df.loc[train_mask].copy()

        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        validation_mask = (
            (df["time"] >= train_end)
            &
            (df["time"] < validation_end)
        )

        validation_df = df.loc[
            validation_mask
        ].copy()

        print()
        print("-" * 70)

        print(
            f"TRAIN: {train_start.date()} -> "
            f"{train_end.date()}"
        )

        print(
            f"VALIDATION: {train_end.date()} -> "
            f"{validation_end.date()}"
        )

        # ----------------------------------------------------
        # Evaluar estrategias
        # ----------------------------------------------------

        fold_results = []

        for strategy in strategy_columns:

            if strategy not in df.columns:

                print(
                    f"[WARNING] No existe: {strategy}"
                )

                continue

            # ------------------------------------------------
            # TRAIN
            # ------------------------------------------------

            try:

                train_resume, train_state = engine(
                    train_df,
                    strategy
                )

            except Exception as e:

                print(
                    f"Error TRAIN {strategy}: {e}"
                )

                continue

            # ------------------------------------------------
            # VALIDATION
            # ------------------------------------------------

            try:

                validation_resume, validation_state = engine(
                    validation_df,
                    strategy
                )

            except Exception as e:

                print(
                    f"Error VALIDATION {strategy}: {e}"
                )

                continue

            # ------------------------------------------------
            # Número de trades
            # ------------------------------------------------

            train_trades = extract_metric(
                train_resume,
                "trades",
                0
            )

            validation_trades = extract_metric(
                validation_resume,
                "trades",
                0
            )

            if train_trades < min_trades:

                continue

            # ------------------------------------------------
            # Scores
            # ------------------------------------------------

            train_score = calculate_strategy_score(
                train_resume
            )

            validation_score = calculate_strategy_score(
                validation_resume
            )

            # ------------------------------------------------
            # Guardar resultado
            # ------------------------------------------------

            result = {

                "train_start": train_start,

                "train_end": train_end,

                "validation_start": train_end,

                "validation_end": validation_end,

                "strategy": strategy,

                "train_score": train_score,

                "validation_score": validation_score,

                "train_return": extract_metric(
                    train_resume,
                    "total_return"
                ),

                "validation_return": extract_metric(
                    validation_resume,
                    "total_return"
                ),

                "train_sharpe": extract_metric(
                    train_resume,
                    "sharpe"
                ),

                "validation_sharpe": extract_metric(
                    validation_resume,
                    "sharpe"
                ),

                "train_drawdown": extract_metric(
                    train_resume,
                    "max_drawdown"
                ),

                "validation_drawdown": extract_metric(
                    validation_resume,
                    "max_drawdown"
                ),

                "train_pf": extract_metric(
                    train_resume,
                    "profit_factor"
                ),

                "validation_pf": extract_metric(
                    validation_resume,
                    "profit_factor"
                ),

                "train_trades": train_trades,

                "validation_trades": validation_trades
            }

            fold_results.append(result)

        # ----------------------------------------------------
        # DataFrame del fold
        # ----------------------------------------------------

        fold_results_df = pd.DataFrame(
            fold_results
        )

        if len(fold_results_df) == 0:

            train_start = (
                train_start
                + pd.DateOffset(months=step_months)
            )

            continue

        # ----------------------------------------------------
        # Seleccionar TOP N
        #
        # IMPORTANTE:
        # La selección se hace con VALIDATION,
        # no con el futuro posterior.
        # ----------------------------------------------------

        fold_results_df = (
            fold_results_df
            .sort_values(
                "validation_score",
                ascending=False
            )
        )

        top_fold = fold_results_df.head(top_n).copy()

        # ----------------------------------------------------
        # Marcar seleccionadas
        # ----------------------------------------------------

        top_fold["selected"] = True

        all_results.append(
            fold_results_df
        )

        selected_strategies.append(
            top_fold
        )

        print()
        print("TOP ESTRATEGIAS")

        print(
            top_fold[
                [
                    "strategy",
                    "train_score",
                    "validation_score",
                    "validation_return",
                    "validation_sharpe",
                    "validation_drawdown",
                    "validation_pf",
                    "validation_trades"
                ]
            ].to_string(index=False)
        )

        # ----------------------------------------------------
        # Avanzar ventana
        # ----------------------------------------------------

        train_start = (
            train_start
            + pd.DateOffset(months=step_months)
        )

    # ========================================================
    # RESULTADOS
    # ========================================================

    if len(all_results) > 0:

        all_results = pd.concat(
            all_results,
            ignore_index=True
        )

    else:

        all_results = pd.DataFrame()

    if len(selected_strategies) > 0:

        selected_strategies = pd.concat(
            selected_strategies,
            ignore_index=True
        )

    else:

        selected_strategies = pd.DataFrame()

    return (
        all_results,
        selected_strategies
    )


# ============================================================
# CONSOLIDAR ESTRATEGIAS SELECCIONADAS
# ============================================================

def consolidate_selected_strategies(
    selected_df,
    top_n=5
):

    """
    Después del Walk Forward tenemos varias ventanas.

    Una estrategia puede aparecer muchas veces.

    Aquí calculamos:

        frecuencia
        score promedio
        validation return promedio
        validation Sharpe promedio

    y seleccionamos las más consistentes.
    """

    if selected_df.empty:

        return pd.DataFrame()

    summary = (
        selected_df
        .groupby("strategy")
        .agg(

            selections=(
                "strategy",
                "count"
            ),

            avg_validation_score=(
                "validation_score",
                "mean"
            ),

            median_validation_score=(
                "validation_score",
                "median"
            ),

            avg_validation_return=(
                "validation_return",
                "mean"
            ),

            avg_validation_sharpe=(
                "validation_sharpe",
                "mean"
            ),

            avg_validation_drawdown=(
                "validation_drawdown",
                "mean"
            ),

            avg_validation_pf=(
                "validation_pf",
                "mean"
            ),

            total_validation_trades=(
                "validation_trades",
                "sum"
            )
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # Score de consistencia
    # --------------------------------------------------------

    summary["consistency_score"] = (

        0.35
        * summary["avg_validation_score"]

        +

        0.25
        * summary["median_validation_score"]

        +

        0.20
        * summary["avg_validation_sharpe"]

        +

        0.20
        * summary["avg_validation_pf"]

    )

    # --------------------------------------------------------
    # Primero estrategias que aparecen muchas veces
    # --------------------------------------------------------

    summary = summary.sort_values(
        [
            "selections",
            "consistency_score"
        ],
        ascending=False
    )

    return summary.head(top_n).reset_index(
        drop=True
    )


# ============================================================
# OOS
# ============================================================

def run_oos(
    df,
    selected_strategies,
    oos_start,
    strategy_columns=None
):

    """
    Ejecuta las estrategias seleccionadas sobre el período OOS.

    IMPORTANTE:

    OOS NO participa en:

        selección
        optimización
        ranking
        ajuste de parámetros

    Es solamente evaluación final.
    """

    df = df.copy()

    # --------------------------------------------------------
    # Validación de fecha
    # --------------------------------------------------------

    if "time" not in df.columns:

        if isinstance(df.index, pd.DatetimeIndex):

            df = df.reset_index()

            if "time" not in df.columns:

                df = df.rename(
                    columns={
                        df.columns[0]: "time"
                    }
                )

        else:

            raise ValueError(
                "El DataFrame debe tener columna 'time'"
            )

    df["time"] = pd.to_datetime(
        df["time"]
    )

    df = (
        df
        .sort_values("time")
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Filtrar OOS
    # --------------------------------------------------------

    oos_df = df.loc[
        df["time"] >= pd.Timestamp(oos_start)
    ].copy()

    if oos_df.empty:

        raise ValueError(
            "El período OOS está vacío."
        )

    # --------------------------------------------------------
    # Estrategias
    # --------------------------------------------------------

    if isinstance(
        selected_strategies,
        pd.DataFrame
    ):

        strategies = (
            selected_strategies[
                "strategy"
            ]
            .drop_duplicates()
            .tolist()
        )

    else:

        strategies = list(
            selected_strategies
        )

    # --------------------------------------------------------
    # Resultados
    # --------------------------------------------------------

    results = []

    states = {}

    trades = {}

    print("=" * 70)
    print("OOS")
    print("=" * 70)

    print(
        "Inicio OOS:",
        oos_df["time"].min()
    )

    print(
        "Fin OOS:",
        oos_df["time"].max()
    )

    print(
        "Estrategias:",
        len(strategies)
    )

    # --------------------------------------------------------
    # Ejecutar cada estrategia
    # --------------------------------------------------------

    for strategy in strategies:

        if strategy not in oos_df.columns:

            print(
                f"[WARNING] No existe: {strategy}"
            )

            continue

        print()
        print(
            f"OOS -> {strategy}"
        )

        try:

            resume, state = engine(
                oos_df,
                strategy
            )

        except Exception as e:

            print(
                f"ERROR OOS {strategy}: {e}"
            )

            continue

        results.append(
            {
                "strategy": strategy,

                "oos_return": extract_metric(
                    resume,
                    "total_return"
                ),

                "oos_sharpe": extract_metric(
                    resume,
                    "sharpe"
                ),

                "oos_sortino": extract_metric(
                    resume,
                    "sortino"
                ),

                "oos_max_drawdown": extract_metric(
                    resume,
                    "max_drawdown"
                ),

                "oos_profit_factor": extract_metric(
                    resume,
                    "profit_factor"
                ),

                "oos_calmar": extract_metric(
                    resume,
                    "calmar"
                ),

                "oos_win_rate": extract_metric(
                    resume,
                    "win_rate"
                ),

                "oos_trades": extract_metric(
                    resume,
                    "trades"
                ),

                "oos_avg_r": extract_metric(
                    resume,
                    "avg_r"
                )
            }
        )

        states[strategy] = state

        # Si tu engine posteriormente devuelve trades:
        #
        # resume, state, trade_df = engine(...)
        #
        # entonces puedes guardar:
        #
        # trades[strategy] = trade_df

    # --------------------------------------------------------
    # DataFrame final
    # --------------------------------------------------------

    results_df = pd.DataFrame(
        results
    )

    if not results_df.empty:

        results_df = (
            results_df
            .sort_values(
                "oos_sharpe",
                ascending=False
            )
            .reset_index(drop=True)
        )

    return (
        results_df,
        states,
        trades
    )


# ============================================================
# COMPARAR WALK FORWARD VS OOS
# ============================================================

def compare_wf_oos(
    wf_summary,
    oos_results
):

    if wf_summary.empty:
        return pd.DataFrame()

    if oos_results.empty:
        return pd.DataFrame()

    comparison = wf_summary.merge(
        oos_results,
        on="strategy",
        how="inner"
    )

    # --------------------------------------------------------
    # Degradación
    #
    # Ejemplo:
    #
    # validation Sharpe = 2
    # OOS Sharpe = 1
    #
    # degradación = 50%
    # --------------------------------------------------------

    comparison["return_degradation"] = (

        comparison["avg_validation_return"]
        - comparison["oos_return"]

    )

    comparison["sharpe_degradation"] = (

        comparison["avg_validation_sharpe"]
        - comparison["oos_sharpe"]

    )

    comparison["pf_degradation"] = (

        comparison["avg_validation_pf"]
        - comparison["oos_profit_factor"]

    )

    # --------------------------------------------------------
    # OOS positivo
    # --------------------------------------------------------

    comparison["oos_positive"] = (

        comparison["oos_return"] > 0

    )

    comparison["oos_sharpe_positive"] = (

        comparison["oos_sharpe"] > 0

    )

    comparison["oos_pf_positive"] = (

        comparison["oos_profit_factor"] > 1

    )

    return comparison